In [1]:
import sys
sys.path.append('D:/Users/imaad/Documents/my-projects/polymarket_crypto')
import sqlite3 as sql3
from utils.config import DB_PATH
from utils.db import get_price
from win_rates import get_win_rates
from datetime import datetime, timedelta
import re

In [38]:
def run_backtest():
    #Step 1: get high win rate wallets (>55%, 100+ positions)
    #get wallets from closed_positions where total > 100 and win_rate > 55
    
    win_rates = get_win_rates()
    good_wallets = [trader for trader in win_rates if (trader['total'] > 100 and trader['win_rate']>90) ]

    # Step 2: get their 5-minute trades only
    #get all closed_positions for those wallets where title has time range (AM-/PM-)

    con = sql3.connect(DB_PATH)
    con.row_factory = sql3.Row
    cur = con.cursor()

    wallets = [w['wallet'] for w in good_wallets]
    placeholders = ','.join(['?' for _ in wallets])
    cur.execute(f"SELECT * FROM closed_positions WHERE wallet IN ({placeholders}) AND (title LIKE '%AM-%' OR title LIKE '%PM-%') AND title LIKE 'Bitcoin%' AND timestamp >= 1770915900 AND avg_price>=0.65", wallets)  

    rows = cur.fetchall()

    trades = [dict(row) for row in rows]
    con.close()

    total_pnl = 0
    num_wins = 0
    num_loss = 0

    for trade in trades:
        title = trade['title']

        time_length = re.search(r'(\d{1,2}:\d{2}[APM]{2})-(\d{1,2}:\d{2}[APM]{2})', title)
        coin_type = re.search(r'^(.*?)\s+Up or Down', title)
        coin_name = coin_type.group(1) if coin_type else None


        binance_ticker = None
        binance_time = int(trade['timestamp']) * 1000

        if coin_name == "Bitcoin":
            binance_ticker = 'BTCUSDT'
        elif coin_name == "Ethereum":
            binance_ticker = 'ETHUSDT'
        elif coin_name == "Solana":
            binance_ticker = 'SOLUSDT'
        elif coin_name == "XRP":
            binance_ticker = 'XRPUSDT'

        start_str, end_str = time_length.groups()
        fmt = "%I:%M%p" 
        
        start_dt = datetime.strptime(start_str, fmt)
        end_dt = datetime.strptime(end_str, fmt)
        
        diff = end_dt - start_dt
        
        if diff.total_seconds() <= 0:
            diff += timedelta(days=1)
            
        minutes = int(diff.total_seconds() / 60)

        binance_entry_price = get_price(binance_ticker, binance_time)[0][3]

        binance_exit_price = get_price(binance_ticker, binance_time+ 300000)[0][3]


        if trade['outcome'] == "Down":
            pnl = binance_exit_price - binance_entry_price
        else:
            pnl = binance_entry_price - binance_exit_price
        
        if pnl >0:
            num_wins +=1
        else:
            num_loss+=1

    
        total_pnl += pnl
    
    num_trades = num_loss+num_wins
    
    return (total_pnl, num_wins/num_trades, num_trades)



   

In [39]:
run_backtest()



(1376.6499999999505, 0.45977011494252873, 696)

In [72]:
a

[(280,
  '0x0ea574f3204c5c9c0cdead90392ea0990f4d17e4',
  '1771563830',
  '0x6b5830015009608f4fafd4b6c5a8b6189689493f8b800001c987b7b6962e7985',
  'Bitcoin Up or Down - February 19, 11:45PM-11:50PM ET',
  'Down',
  0.766911,
  1117.948739,
  5),
 (281,
  '0x0ea574f3204c5c9c0cdead90392ea0990f4d17e4',
  '1771563830',
  '0xfcc2ef95399bde56af12a872d1385361db67d526ee214ea6b726f9958f1a28ff',
  'Bitcoin Up or Down - February 19, 11:45PM-12:00AM ET',
  'Down',
  0.653388,
  424.679187,
  15),
 (282,
  '0x0ea574f3204c5c9c0cdead90392ea0990f4d17e4',
  '1771563830',
  '0xfcc2ef95399bde56af12a872d1385361db67d526ee214ea6b726f9958f1a28ff',
  'Bitcoin Up or Down - February 19, 11:45PM-12:00AM ET',
  'Up',
  0.327928,
  -81.80949200000002,
  15),
 (283,
  '0x0ea574f3204c5c9c0cdead90392ea0990f4d17e4',
  '1771563830',
  '0xad41b8221921c14c8abbe73d828756b93d2b6ad7eec64b37fd7a4732fb303334',
  'Bitcoin Up or Down - February 19, 11:50PM-11:55PM ET',
  'Up',
  0.237119,
  -725.910007,
  5),
 (284,
  '0x0ea574f3